# 11: Train the image quality model

Trains an EfficientNet-B0 on the EyeQ quality labels (good / usable / reject) of the EyePACS
`train` split, with class-weighted cross-entropy, and keeps the epoch with the best `val`
macro-F1. Then the selected model predicts every processed image of the evaluation splits of all
sites, and the metrics are computed: EyeQ internal test, and DDR images marked ungradable.

**Settings** (right-hand panel): Accelerator **GPU T4 x1**, Internet **On**,
Persistence **Files only**.
**Inputs** (*Add Input*):
- *Notebooks* → *Your Work* → **01**, **02**, **03** and **04** (processed image shards of
  EyePACS, APTOS, IDRiD and DDR)
- *Datasets* → *Your Datasets* → **retinaguard-manifest** (`manifest.parquet`, private)
- To **resume** a stopped run: also the output of this notebook's previous version.

**Run:** *Save Version* → *Save & Run All (Commit)*. At most 15 epochs, early stop after 4
without a better validation macro-F1.

**Output** (`/kaggle/working/runs/quality-efficientnet_b0/`): `best.pt`, `last.pt`,
`metrics.jsonl`, `params.json`, `tags.json`, `config.yaml`, `curves.png`,
`predictions.parquet` and `quality_metrics.csv`.

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
OVERRIDES = []  # Hydra overrides, e.g. ["train.batch_size=16"]

In [ ]:
EXTRAS = '"pandera>=0.20" "hydra-core>=1.3" "timm>=1.0" "albumentations>=2.0"'
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" {EXTRAS}

In [ ]:
# --- 1. Inputs, configuration and an earlier run to resume -----------------------
import shutil
import subprocess
from pathlib import Path

import torch

from retinaguard import __version__
from retinaguard.config import load_config
from retinaguard.tracking import RunLogger

INPUT_ROOT = Path("/kaggle/input")
RUNS_DIR = Path("/kaggle/working/runs")
manifests = sorted(INPUT_ROOT.glob("**/manifest.parquet"))
assert manifests, "manifest.parquet not found: attach the retinaguard-manifest dataset"
shard_dirs = sorted({p.parent for p in INPUT_ROOT.glob("**/*_[0-9][0-9][0-9].zip")})
for prefix in ("eyepacs", "aptos", "idrid", "ddr"):
    assert any(d.glob(f"{prefix}_*.zip") for d in shard_dirs), f"{prefix} shards not found"

cfg = load_config(
    "train/quality",
    [
        f"output_dir={RUNS_DIR}",
        f"data.manifest={manifests[0]}",
        f"data.shard_dirs=[{','.join(str(d) for d in shard_dirs)}]",
        *OVERRIDES,
    ],
)
run_dir = RUNS_DIR / cfg.run_name
earlier = sorted(INPUT_ROOT.glob(f"**/runs/{cfg.run_name}/last.pt"))
if earlier and not (run_dir / "last.pt").exists():
    shutil.copytree(earlier[0].parent, run_dir, dirs_exist_ok=True)
    print("resuming from", earlier[0].parent)

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("\t")[0]
assert torch.cuda.is_available(), (
    f"no GPU (torch {torch.__version__}): set Accelerator to GPU in the notebook settings, "
    "then save a new version"
)
gpu = torch.cuda.get_device_name(0)
RunLogger(run_dir).set_tags({"git_sha": git_sha, "gpu": gpu, "platform": "kaggle"})
print(f"retinaguard {__version__} @ {git_sha[:10]}, torch {torch.__version__}, GPU {gpu}")
print("shards:", [str(d) for d in shard_dirs])

In [ ]:
# --- 2. Train (or continue) ----------------------------------------------------
from retinaguard.train.train_quality import train

run_dir = train(cfg)

In [ ]:
# --- 3. Learning curves and the selected epoch ---------------------------------
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.DataFrame(RunLogger(run_dir).read_metrics())
table = metrics.pivot_table(index="step", columns="key", values="value", aggfunc="last")
columns = ["train_loss", "val_loss", "val_macro_f1", "val_recall_reject", "epoch_minutes"]
print(table[columns].round(4).to_string())
best = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
print(f"best epoch {best['epoch']}: val macro-F1 {best['val_macro_f1']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
table[["train_loss", "val_loss"]].plot(ax=axes[0], title="loss")
table[["val_macro_f1", "val_recall_reject"]].plot(ax=axes[1], title="validation")
for ax in axes:
    ax.set_xlabel("epoch")
plt.tight_layout()
fig.savefig(run_dir / "curves.png", dpi=80)
plt.show()

In [ ]:
# --- 4. Predict every evaluation image, then measure ---------------------------
from retinaguard.train.evaluate_quality import main as evaluate_main
from retinaguard.train.train_quality import predict_run

predictions = predict_run(cfg, run_dir)
print(f"{len(predictions):,} predictions")
evaluate_main(
    [
        f"--manifest={manifests[0]}",
        f"--predictions={run_dir / 'predictions.parquet'}",
        f"--out={run_dir / 'quality_metrics.csv'}",
    ]
)

**After the version finishes:** keep the version; its output holds the predictions used by
the triage policy. Do not download the `.pt` files into the repository.